# Reconstitution du vrai test -- SOLUTIONS

Chaque question : solution complete + explication du raisonnement (pourquoi
cette approche, pieges specifiques a CE type de question BCG X).


---
## Question 1 -- Data analysis -- Solution


In [ ]:
import numpy as np
import pandas as pd

DATA = "../data/q1"

drivers = pd.read_csv(f"{DATA}/drivers.csv")
rides = pd.concat(
    [pd.read_csv(f"{DATA}/rides_{i}.csv") for i in range(1, 5)],
    ignore_index=True,
)

average_driver_rating = drivers["rating"].mean()
percentage_drivers_with_second_language = (
    (drivers["second_language"] != "no").mean() * 100
)
ride_success_rate = (rides["status"] == "Success").mean() * 100

results = pd.DataFrame({
    "insight_type": [
        "average_driver_rating",
        "percentage_drivers_with_second_language",
        "ride_success_rate",
    ],
    "value": [
        average_driver_rating,
        percentage_drivers_with_second_language,
        ride_success_rate,
    ],
})
results.to_csv("analysis_results.csv", index=False)
results


**COMMENT RAISONNER FACE A CE TYPE DE QUESTION.** Trois "insights"
independants, chacun mappe sur une ligne d'un CSV a deux colonnes
(`insight_type`, `value`) -- c'est le format canonique de la Q1 BCG X:
peu importe le nombre d'insights demandes, la structure de sortie ne
change jamais. Traitez chaque insight comme un mini-calcul isole, puis
assemblez a la fin.

* **Moyenne simple** -> `drivers["rating"].mean()`, rien de plus.
* **Pourcentage d'une condition** -> le reflexe `condition.mean() * 100`
  (moyenne d'un booleen = taux). Ici `second_language != "no"`.
* **Combiner 4 fichiers avant de calculer** -> `pd.concat([...],
  ignore_index=True)`. Piege frequent : calculer le taux de succes
  fichier par fichier puis faire la moyenne des 4 taux -- **faux** si les
  fichiers n'ont pas le meme nombre de lignes (moyenne non ponderee).
  Toujours concatener d'abord, calculer ensuite.

**SCHEMA A RECONNAITRE.** *"Data analysis, plusieurs insights, CSV
insight_type/value"* -> calculer chaque insight separement sur les
donnees deja nettoyees/combinees, assembler en DataFrame a la fin,
`to_csv(index=False)`.


---
## Question 2 -- Data collecting -- Solution


In [ ]:
DATA2 = "../data/q2"

drivers2 = pd.read_csv(f"{DATA2}/drivers.csv")
cars = pd.read_csv(f"{DATA2}/cars.csv")
rides2 = pd.concat(
    [pd.read_csv(f"{DATA2}/rides_{i}.csv") for i in range(1, 5)],
    ignore_index=True,
)

TODAY = pd.Timestamp("2023-04-15")

# --- upvotes per driver -------------------------------------------------
upvote_cols = [
    "car_clearness_upvote_given", "politeness_upvote_given",
    "communication_upvote_given", "punctuality_upvote_given",
]
rides2["n_upvotes_this_ride"] = rides2[upvote_cols].sum(axis=1)
upvotes_per_driver = (
    rides2.groupby("driver_id")["n_upvotes_this_ride"].sum()
    .rename("number_of_upvotes")
)

# --- car info -------------------------------------------------------
cars["last_inspection_date"] = pd.to_datetime(cars["last_inspection_date"])
cars["days_since_inspection"] = (TODAY - cars["last_inspection_date"]).dt.days

# --- assemble -------------------------------------------------------
collected = drivers2.merge(
    cars[["car_id", "model", "manufacture_year", "days_since_inspection"]],
    on="car_id", how="left",
).rename(columns={"model": "car_model", "manufacture_year": "car_manufacture_year"})

collected["experience"] = 2023 - collected["started_driving_year"]

collected = collected.merge(upvotes_per_driver, on="driver_id", how="left")
collected["number_of_upvotes"] = collected["number_of_upvotes"].fillna(0).astype(int)

collected = collected[[
    "driver_id", "car_model", "car_manufacture_year", "days_since_inspection",
    "age", "experience", "second_language", "rating", "net_worth_of_tips",
    "number_of_upvotes", "driver_class",
]]
collected.to_csv("collected.csv", index=False)
collected.head()


**COMMENT RAISONNER FACE A CE TYPE DE QUESTION.** C'est un exercice de
**jointure + agregation + derivation de colonnes**, pas de nettoyage. La
methode generale, valable pour toute Q2 de ce style :

1. **Repartir du schema de sortie demande**, colonne par colonne, et
   identifier sa source : vient-elle directement d'une table
   (`age`, `rating`), d'une jointure (`car_model` via `cars`), d'un calcul
   simple (`experience = 2023 - started_driving_year`), ou d'une
   agregation sur une table d'evenements (`number_of_upvotes` = somme sur
   toutes les courses du chauffeur) ?
2. **Sommer plusieurs colonnes booleennes en une ligne** ->
   `df[bool_cols].sum(axis=1)` donne le nombre de `True` par ligne (ici,
   par course) ; agreger ensuite par chauffeur avec `groupby().sum()`.
3. **"Nombre de jours depuis une date"** -> toujours passer par
   `pd.to_datetime`, puis soustraire d'une date de reference (`TODAY`
   fournie par l'enonce -- ici 2023-04-15, ne jamais utiliser
   `pd.Timestamp.now()`) et prendre `.dt.days`.
4. **`experience`** est une formule explicite donnee dans l'enonce
   (`2023 - started_driving_year`) -- ne pas la recalculer autrement
   (par exemple via une difference de dates), meme si le resultat
   parait equivalent : suivez la formule a la lettre.
5. **fillna(0) sur `number_of_upvotes`** : un chauffeur sans aucune
   course n'a pas de ligne dans `rides`, donc le merge produit un `NaN`
   -- logiquement, 0 courses = 0 upvote.
6. **Tests order-agnostic** -- l'enonce dit explicitement que l'ordre des
   lignes/colonnes n'importe pas : pas besoin de trier, gagnez du temps
   ailleurs.

**SCHEMA A RECONNAITRE.** *"Data collecting, table de sortie avec colonnes
venant de plusieurs fichiers"* -> partir du schema cible, mapper chaque
colonne a sa source (directe / jointure / derivee / agregee), assembler
avec des `merge` successifs.


In [ ]:
# sauvegarde une copie de reference pour que la Q3 de ce notebook
# puisse repartir d'un collected.csv sans devoir re-executer Q2
import os
os.makedirs("../data/q2", exist_ok=True)
collected.to_csv("../data/q2/collected_reference.csv", index=False)


---
## Question 3 -- Data processing -- Solution


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

train_raw, test_raw = train_test_split(collected, test_size=0.30, random_state=42)
train_raw.to_csv("../data/train.csv", index=False)
test_raw.to_csv("../data/test.csv", index=False)

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# a. age -> moyenne du TRAIN uniquement, arrondie a l'entier ------------
mean_age = round(train["age"].mean())
train["age"] = train["age"].fillna(mean_age)
test["age"] = test["age"].fillna(mean_age)          # meme valeur, pas de fuite

# b. ordinal encoding, fit sur TRAIN uniquement --------------------------
cat_cols = ["second_language", "car_model"]
encoder = OrdinalEncoder(dtype=int)
train[cat_cols] = encoder.fit_transform(train[cat_cols])
test[cat_cols] = encoder.transform(test[cat_cols])

# c. standard scaling, fit sur TRAIN uniquement --------------------------
scaler = StandardScaler()
train["net_worth_of_tips"] = scaler.fit_transform(train[["net_worth_of_tips"]]).round(5)
test["net_worth_of_tips"] = scaler.transform(test[["net_worth_of_tips"]]).round(5)

# d. driver_class binaire -------------------------------------------------
mapping = {"A class": 0, "B class": 1}
train["driver_class"] = train["driver_class"].map(mapping)
test["driver_class"] = test["driver_class"].map(mapping)

train["net_worth_of_tips"] = train["net_worth_of_tips"].map(lambda x: f"{x:.5f}")
test["net_worth_of_tips"] = test["net_worth_of_tips"].map(lambda x: f"{x:.5f}")

train.to_csv("processed_train.csv", index=False)
test.to_csv("processed_test.csv", index=False)
train.head()


**COMMENT RAISONNER FACE A CE TYPE DE QUESTION.** C'est un exercice de
**preprocessing ML classique**, note etape par etape -- traitez chaque
sous-question (a/b/c/d) comme un bloc independant que vous pouvez valider
separement, plutot que d'ecrire tout le pipeline d'un bloc.

**a. `fillna` avec la moyenne -- LA REGLE DE FUITE CRITIQUE.** La moyenne
doit etre calculee **uniquement sur `train`**, puis appliquee telle
quelle (comme constante) a `test`. Si vous calculez `test["age"].mean()`
separement, vous laissez de l'information du test influencer le
pretraitement -- c'est une fuite de donnees. La regle generale : **toute
statistique apprise sur les donnees (moyenne, encodage, parametres de
scaling) vient du train, jamais du test.**

**b. Encodage ordinal -- LE PIEGE DU MAPPING NON-CONSECUTIF.** L'enonce
donne volontairement un exemple correct ET un exemple incorrect : la
seule difference est que le mapping incorrect n'est **ni consecutif ni
demarre a zero**. `sklearn.preprocessing.OrdinalEncoder` garantit cela
nativement (codes toujours `0, 1, 2, ...` par ordre alphabetique des
categories vues au `fit`). Le piege classique est d'ecrire un mapping
manuel a la main (`{"Nissan Altima": 3, ...}`) sans verifier qu'il
respecte cette contrainte, ou de `fit` l'encodeur separement sur train
et sur test (categories vues dans des ordres differents -> mappings
incoherents entre les deux fichiers).

**c. Standard Scaling.** `StandardScaler` doit etre **fit sur train
seul**, puis seulement `.transform()` (jamais `.fit_transform()`) sur
test -- sinon vous utilisez la moyenne/variance du test pour standardiser
le test, ce qui est a la fois une fuite et incoherent avec le
pretraitement applique au train.

**d. Le piege des 5 decimales EXACTES.** L'enonce demande "exactly 5
decimals" -- un `round(5)` seul ne suffit pas si vous ecrivez ensuite le
CSV avec `to_csv` sans specifier le format (`1.2` s'ecrirait `1.2`, pas
`1.20000`). Formatez explicitement la colonne en chaine avec
`s.map(lambda x: f"{x:.5f}")` **avant** `to_csv`. Attention au piege
inverse : `to_csv(..., float_format="%.5f")` s'applique a **toutes** les
colonnes flottantes du DataFrame, pas seulement `net_worth_of_tips` --
ca reformaterait aussi `rating` en `4.21000`, ce qui n'est pas demande.
Cibler uniquement la colonne concernee evite cet effet de bord.

**SCHEMA A RECONNAITRE.** *"Data processing avec contrainte no
leakage"* -> systematiquement `fit`/apprentissage sur train uniquement,
`transform` (jamais `fit_transform`) sur test ; toute statistique de
remplissage vient du train.


---
## Question 4 -- Classification -- Solution


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, confusion_matrix

full = pd.concat([train, test], ignore_index=True)   # les deux deja pretraites (Q3)

tr, va_te = train_test_split(full, test_size=0.30, random_state=42, stratify=full["driver_class"])
va, te = train_test_split(va_te, test_size=0.50, random_state=42, stratify=va_te["driver_class"])

tr.to_csv("../data/train.csv", index=False)
va.to_csv("../data/val.csv", index=False)
te.drop(columns=["driver_class"]).to_csv("../data/test.csv", index=False)   # test SANS labels

X_tr, y_tr = tr.drop(columns=["driver_class"]), tr["driver_class"]
X_va, y_va = va.drop(columns=["driver_class"]), va["driver_class"]
X_te = te.drop(columns=["driver_class"])

# class_weight="balanced" pour pousser le recall de la classe minoritaire (B class)
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_tr, y_tr)

# ajuster le seuil de decision pour privilegier le recall sans sacrifier
# completement la precision -- on cherche le seuil qui maximise le recall
# sous une contrainte de precision minimale (ex: >= 0.6)
proba_va = model.predict_proba(X_va)[:, 1]
best_threshold, best_recall = 0.5, -1
for thr in np.arange(0.1, 0.9, 0.02):
    pred = (proba_va >= thr).astype(int)
    prec = precision_score(y_va, pred, zero_division=0)
    rec = recall_score(y_va, pred, zero_division=0)
    if prec >= 0.6 and rec > best_recall:
        best_threshold, best_recall = thr, rec

pred_va = (proba_va >= best_threshold).astype(int)
print("seuil retenu:", best_threshold)
print("precision (val):", precision_score(y_va, pred_va))
print("recall (val):", recall_score(y_va, pred_va))
print(confusion_matrix(y_va, pred_va))

proba_te = model.predict_proba(X_te)[:, 1]
pred_te = (proba_te >= best_threshold).astype(int)
pd.DataFrame({"driver_class": pred_te}).to_csv("predictions.csv", index=False)


**COMMENT RAISONNER FACE A CE TYPE DE QUESTION.** La contrainte cle est
verbale et facile a rater si on code vite : *"maximize recall, while
keeping precision on the relatively high level"*. Ce n'est **pas** "un
bon F1" -- c'est un objectif asymetrique. Deux leviers, a utiliser
ensemble :

1. **`class_weight="balanced"`** (ou un `class_weight` manuel favorisant
   la classe positive) au moment du `fit` -- le modele penalise plus
   fortement les faux negatifs sur `B class`, donc apprend a mieux la
   detecter.
2. **Ajuster le seuil de decision** plutot que d'utiliser le defaut de
   `predict()` (seuil 0.5 implicite). En balayant les seuils sur `proba`
   et en gardant celui qui maximise le recall **sous une contrainte
   minimale de precision**, on traduit exactement la consigne de
   l'enonce en code -- au lieu de choisir un seuil au hasard.

**Pourquoi évaluer sur `val.csv` et pas sur `test.csv`.** Le test ne
contient pas les labels dans le vrai test -- c'est litteralement
impossible d'y calculer precision/recall en local. Le val set est fait
pour ça : itérer sur le modele/seuil, puis, une fois satisfait,
predire sur test et soumettre.

**PIEGE FREQUENT.** Refaire un split 70/15/15 sur les donnees deja
pretraitees (Q3) est correct **seulement** si l'encodage/scaling (fit
sur l'ancien train) reste valable pour le nouveau decoupage -- dans le
vrai test, les fichiers train/val/test de Q4 sont deja fournis
pretraites, donc cette etape n'est pas a refaire ; ici on la simule
uniquement parce qu'on reutilise nos propres fichiers Q3 comme source.

**SCHEMA A RECONNAITRE.** *"Maximiser recall en gardant precision
elevee, classe positive minoritaire"* -> `class_weight="balanced"` +
recherche de seuil sur `predict_proba` avec contrainte de precision
minimale, valide sur le validation set, jamais sur le test (qui n'a pas
les labels).
